#!/usr/bin/env python3
# -*- coding: utf-8 -*-

================================================================================
 Apply FIXED winning configurations (rule, p) -- one per system, already
 selected from TRAIN -- to the 5-fold held-out TEST (20%) subsets.

 No p sweep here: each system uses exactly the single (rule, p) pair that
 won on TRAIN. This is the honest generalization check -- the winners were
 chosen without ever looking at the test folds; this script is the first
 time the test folds are touched.

 Output columns: Fold | System | Rule | p | EER (%) | Rank-1 @ FAR=1% | N_test
================================================================================

This script reuses the normalization, fusion-rule, metric, and fold-splitting
building blocks from evaluate_pdep_train_5fold.py, so the TEST evaluation is
guaranteed consistent with how TRAIN was evaluated (same seed -> same folds,
same per-fold min-max fit on TRAIN only, same EER/Rank-1 definitions).

USAGE
-----
    python evaluate_best_configs_on_test.py --seed 62

    # Or supply the winning configs explicitly (skip re-deriving them):
    python evaluate_best_configs_on_test.py --seed 62 \\
        --configs-csv Overall_Best_per_System.csv

BEST_CONFIGS below is the default, taken from the seed=62 TRAIN run already
performed. Edit it directly, or pass --configs-csv to read it from a file
produced by evaluate_pdep_train_5fold.py's "Overall_Best_per_System" sheet.
================================================================================

In [ ]:
from __future__ import annotations

import argparse
import sys
from pathlib import Path
from typing import Dict, List, Tuple

from scipy.stats import wilcoxon
from openpyxl import load_workbook
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

#sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path.cwd()))

from evaluate_pdep_train_5fold import (  # noqa: E402
    DataConfig,
    SplitConfig,
    SYSTEM_DEFINITIONS,
    build_system_matrix,
    compute_eer,
    compute_rank1_at_far,
    make_folds,
    make_normalizer,
    split_genuine_impostor,
    submatrix,
    train_indices_for_fold,
)


# ==============================================================================
#  CONFIGURATION
# ==============================================================================

DEFAULT_MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"
SEED = 62
N_FOLDS = 5
FAR_TARGET = 0.01

# The winning (rule, p) per system, as selected on TRAIN by
# evaluate_pdep_train_5fold.py (seed=62, p step=0.05). Override with
# --configs-csv to load a different set instead of editing this table.
BEST_CONFIGS: Dict[str, Tuple[str, float]] = {
    "S1": ("R5p",        0.4),
    "S2": ("Werners1",   0.45),
    "S3": ("R5p",        0.25),
    "S4": ("R5p",        0.30),
    "S5": ("Luhandjula", 0.45),
    "S6": ("Werners2",   0.45),
    "S7": ("Mizumoto2",  0.75),
    "S8": ("Luhandjula", 0.8),
}

# Fusion system → unimodal baselines to compare against
BASELINE_MAP: Dict[str, List[str]] = {
    "S1": ["Face_C", "Face_G"],
    "S2": ["Face_C", "Finger_LI"],
    "S3": ["Face_C", "Finger_RI"],
    "S4": ["Face_G", "Finger_LI"],
    "S5": ["Face_G", "Finger_RI"],
    "S6": ["Finger_LI", "Finger_RI"],
    "S7": ["Face_C", "Finger_LI", "Finger_RI"],
    "S8": ["Face_G", "Finger_LI", "Finger_RI"],
}


# ==============================================================================
#  STEP 1 -- EVALUATE EACH FIXED CONFIG ON ITS FOLD'S HELD-OUT TEST SET
# ==============================================================================

def evaluate_best_configs_on_test(mat_file: str,
                                   seed: int,
                                   n_folds: int,
                                   best_configs: Dict[str, Tuple[str, float]],
                                   far_target: float = FAR_TARGET) -> pd.DataFrame:
    """
    For every fold and every system, apply that system's single fixed
    (rule, p) -- unchanged across all folds -- to the fold's TEST submatrix.

    Normalization is fit on the fold's TRAIN submatrix only (same as during
    training) and applied to TEST, so there is no leakage: the p sweep never
    touches test data, and neither does the normalization scale.
    """
    from scipy.io import loadmat

    data_cfg = DataConfig(mat_file=mat_file)
    split_cfg = SplitConfig(n_folds=n_folds, seed=seed, shuffle=True)

    contents = loadmat(mat_file)
    raw_matrices = {
        modality: np.asarray(contents[key], dtype=float)
        for modality, key in data_cfg.matrix_keys.items()
    }
    n_subjects = next(iter(raw_matrices.values())).shape[0]
    folds = make_folds(n_subjects, split_cfg)

    records: List[dict] = []
    for fold_idx in range(n_folds):
        test_ids = folds[fold_idx]
        train_ids = train_indices_for_fold(folds, fold_idx, n_subjects)

        # Fit normalization on TRAIN, apply to TEST -- same protocol as training.
        normalized_test: Dict[str, np.ndarray] = {}
        for modality, matrix in raw_matrices.items():
            train_block = submatrix(matrix, train_ids)
            test_block = submatrix(matrix, test_ids)
            normalizer = make_normalizer(data_cfg, label=modality)
            normalizer.fit(train_block)
            normalized_test[modality] = normalizer.transform(test_block)

        for system_name, (rule, p) in best_configs.items():
            system = SYSTEM_DEFINITIONS[system_name]
            fused_test = build_system_matrix(
                system, normalized_test, rule, p, data_cfg.invalid_value
            )
            genuine, impostor = split_genuine_impostor(fused_test, data_cfg.invalid_value)

            records.append({
                "Fold": fold_idx + 1,
                "System": system_name,
                "Rule": rule,
                "p": p,
                "EER (%)": round(compute_eer(genuine, impostor) * 100, 4),
                "Rank-1 @ FAR=1%": round(
                    compute_rank1_at_far(fused_test, far_target, data_cfg.invalid_value) * 100,
                    4,
                ),
                "N_test": len(test_ids),
            })

        print(f"  Fold {fold_idx + 1}/{n_folds} done "
              f"({len(test_ids)} test subjects).")

    report = pd.DataFrame.from_records(records)
    system_order = list(best_configs.keys())
    report["System"] = pd.Categorical(report["System"], system_order, ordered=True)
    return report.sort_values(["System", "Fold"]).reset_index(drop=True)

In [ ]:
# ==============================================================================
#  STEP 2 -- WRITE THE FORMATTED, SEED-TAGGED XLSX
# ==============================================================================

def write_report(report: pd.DataFrame, seed: int, n_folds: int,
                 best_configs: Dict[str, Tuple[str, float]], output_dir: Path) -> Path:
    font_name = "Arial"
    theme_color = "B45309"  # distinct color from the TRAIN report (orange, not blue)

    header_fill = PatternFill("solid", fgColor=theme_color)
    header_font = Font(name=font_name, bold=True, color="FFFFFF", size=10)
    title_font = Font(name=font_name, bold=True, size=13, color=theme_color)
    subtitle_font = Font(name=font_name, italic=True, size=9, color="555555")
    cell_font = Font(name=font_name, size=10)
    border = Border(*(Side(style="thin", color="D9D9D9") for _ in range(4)))

    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "Results"

    configs_text = "; ".join(f"{s}={r}(p={p:.2f})" for s, (r, p) in best_configs.items())

    worksheet.cell(row=1, column=1,
                   value=f"Fixed BEST_CONFIGS applied to 5-Fold held-out TEST (20%), "
                         f"seed={seed}").font = title_font
    worksheet.cell(row=2, column=1,
                   value=f"{len(best_configs)} systems x {n_folds} disjoint folds = "
                         f"{len(report)} rows. NO p sweep: each system uses exactly one "
                         "fixed (rule, p), selected on TRAIN and never re-tuned on TEST. "
                         "Normalization fit on each fold's TRAIN submatrix, applied to "
                         "that fold's TEST submatrix.").font = subtitle_font
    worksheet.cell(row=3, column=1, value=f"Configurations used: {configs_text}").font = subtitle_font

    header_row = 5
    for col_idx, column in enumerate(report.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=column)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row_offset, record in enumerate(report.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            column = report.columns[col_idx - 1]
            cell = worksheet.cell(row=header_row + row_offset, column=col_idx, value=value)
            cell.font = cell_font
            cell.border = border
            if column == "p":
                cell.number_format = "0.00"
            elif column in ("EER (%)", "Rank-1 @ FAR=1%"):
                cell.number_format = "0.0000"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit_columns(worksheet, len(report.columns), header_row, len(report))

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"PDependent_BestConfigs_TEST_seed{seed}.xlsx"
    workbook.save(output_path)
    return output_path


def _autofit_columns(worksheet, n_columns: int, header_row: int,
                     n_data_rows: int, minimum: int = 10, maximum: int = 30) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_data_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)

In [2]:
# ----------------------------------------------------------------------
# Entry point
# ----------------------------------------------------------------------
if __name__ == "__main__":
    print('-' * 48)
    best_configs = BEST_CONFIGS

    print(f"Applying {len(best_configs)} fixed (rule, p) configs to the TEST folds "
          f"(seed={SEED}, {N_FOLDS} folds) -- no sweep.")
    for system, (rule, p) in best_configs.items():
        print(f"  {system}: {rule} (p={p:.2f})")

    DEFAULT_MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"

    report = evaluate_best_configs_on_test(
       DEFAULT_MAT_FILE , SEED, N_FOLDS, BEST_CONFIGS, FAR_TARGET
    )

    output_path = write_report(report, SEED, N_FOLDS, BEST_CONFIGS, Path("./Results"))
    print(f"\nReport written to: {output_path}")
    print(report.to_string(index=False))


    
    print("-" * 48)
    print("done\t" * 3)
    print("-" * 48)

------------------------------------------------
Applying 8 fixed (rule, p) configs to the TEST folds (seed=62, 5 folds) -- no sweep.
  S1: R5p (p=0.40)
  S2: Werners1 (p=0.45)
  S3: R5p (p=0.25)
  S4: R5p (p=0.30)
  S5: Luhandjula (p=0.45)
  S6: Werners2 (p=0.45)
  S7: Mizumoto2 (p=0.75)
  S8: Luhandjula (p=0.80)
  Fold 1/5 done (104 test subjects).
  Fold 2/5 done (104 test subjects).
  Fold 3/5 done (103 test subjects).
  Fold 4/5 done (103 test subjects).
  Fold 5/5 done (103 test subjects).

Report written to: Results\PDependent_BestConfigs_TEST_seed62.xlsx
 Fold System       Rule    p  EER (%)  Rank-1 @ FAR=1%  N_test
    1     S1        R5p 0.40   1.9231          98.0769     104
    2     S1        R5p 0.40   4.8077          87.5000     104
    3     S1        R5p 0.40   5.8824          93.1373     103
    4     S1        R5p 0.40   3.0935          89.3204     103
    5     S1        R5p 0.40   1.9084          87.3786     103
    1     S2   Werners1 0.45   0.9615          99.038